# Enrich Court Authority Cards — Qwen3.5-35B-A3B with vLLM

Enriches `court_authority_cards_v4.jsonl` with LLM-derived semantic fields for RAG.

**Model:** `Qwen/Qwen3.5-35B-A3B`  
**Runtime:** vLLM offline inference (`LLM.generate`)  
**GPU target:** single high-VRAM NVIDIA GPU, e.g. RTX PRO 6000 Blackwell / G4-class 96 GB VRAM  
**Structured output:** vLLM `StructuredOutputsParams(json=RAG_SCHEMA)` when available, with a compatibility fallback for older vLLM builds.  
**Thinking mode:** disabled at chat-template rendering time with `enable_thinking=False`.

The notebook is vLLM-only. It does not use Hugging Face `model.generate()`, `torch.no_grad()`, or manual tokenization for inference.


## 1 · Environment setup

In [2]:
import sys, os

IN_COLAB = 'google.colab' in sys.modules
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    # Verify GPU
    import subprocess
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                            capture_output=True, text=True)
    print('GPU:', result.stdout.strip())

Running in Colab: True
GPU: NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


In [ ]:
if IN_COLAB:
    # The vLLM Qwen3.5 recipe recommends installing vLLM with the matching Torch backend.
    # In Colab notebooks, using uv with --system is the closest equivalent to:
    #   uv pip install -U vllm --torch-backend=auto
    %pip install -q -U uv tqdm
    !uv pip install --system -U vllm --torch-backend=auto


In [3]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Configuration

In [ ]:
from pathlib import Path

# ── Paths ────────────────────────────────────────────────────────────────────
BASE_DIR    = Path('/content/drive/MyDrive/swiss_law') if IN_COLAB else Path('..').resolve()
DATA_DIR    = BASE_DIR / 'data'
INSIGHTS_DIR= BASE_DIR / 'data_insights'
ART_DIR     = BASE_DIR / 'artifacts'
SCRIPT_DIR  = BASE_DIR / 'scripts'

for d in [DATA_DIR, INSIGHTS_DIR, ART_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Model ────────────────────────────────────────────────────────────────────
# Official Qwen3.5-35B-A3B chat / multimodal checkpoint.
# The HF repo is ~71.9 GB of safetensors. On a ~96 GB GPU, keep context and
# batch size conservative for the first run, then increase after a clean load.
MODEL_ID     = 'Qwen/Qwen3.5-35B-A3B'
QUANTIZATION = None     # None for BF16. Use a vLLM-supported quantized checkpoint separately.

# ── Inference ────────────────────────────────────────────────────────────────
GPU_MEMORY_UTIL    = 0.90   # Start conservative on 95.8 GB VRAM; raise to 0.92 only if stable.
MAX_MODEL_LEN      = 4096
BATCH_SIZE         = 4      # Safe first-run value. Try 8/16 after confirming no OOM.
TEMPERATURE        = 0.15   # Lower = more deterministic JSON.
MAX_TOKENS         = 600
TENSOR_PARALLEL    = 1      # Single GPU; set 2+ only on a multi-GPU runtime.

# vLLM recipe: latency-focused mode uses MTP-1 speculative decoding and disables prefix caching.
USE_MTP            = True
MTP_TOKENS         = 1
ENABLE_PREFIX_CACHING = False

# Text-only workload. If the installed vLLM exposes the Python equivalent of
# --language-model-only, the loader cell will enable it automatically.
LANGUAGE_MODEL_ONLY = True

# ── Files ────────────────────────────────────────────────────────────────────
INPUT_FILE      = ART_DIR / 'court_authority_cards_v4.jsonl'
OUTPUT_FILE     = ART_DIR / 'court_authority_cards_rag.jsonl'
CHECKPOINT_FILE = ART_DIR / 'rag_checkpoint.txt'

# Optional: stop after N cards (0 = process all)
LIMIT = 1000

print('BASE_DIR   :', BASE_DIR)
print('INPUT_FILE :', INPUT_FILE)
print('OUTPUT_FILE:', OUTPUT_FILE)
print('MODEL_ID   :', MODEL_ID)


## 3 · JSON schema + prompts

In [ ]:
import re

# ── Pre-filter: trivial paragraphs that don't need an LLM ────────────────────
COST_PROC_RE = re.compile(
    r'(?:'
    r'\bgerichtskosten\b|\bprozesskosten\b|\bverfahrenskosten\b|'
    r'\bfrais judiciaires\b|\bfrais de la cause\b|\bd[eé]pens\b|'
    r'\bspese giudiziarie\b|\bripetibili\b|'
    r'\bparteientsch[äa]digung\b|\bhonoraire\b|'
    r'\bunentgeltliche rechtspflege\b|\bassistance judiciaire\b|'
    r'\bpatrocinio gratuito\b|'
    r'\bdie sache wird .{0,80}zur[üu]ckgewiesen\b|'
    r'\brenvoyer la cause\b|'
    r'\bla causa [eè] rinviata\b|'
    r'^\s*\d+\.\s*\d+\..{0,5}fr\.\s*\d'
    r')',
    re.IGNORECASE | re.MULTILINE,
)

# ── JSON schema for guided generation ────────────────────────────────────────
RAG_SCHEMA = {
    'type': 'object',
    'properties': {
        'english_summary':          {'type': 'string'},
        'legal_topic':              {'type': 'string'},
        'legal_question':           {'type': 'string'},
        'legal_rule':               {'type': 'string'},
        'court_holding':            {'type': 'string'},
        'factual_context':          {'type': 'string'},
        'english_legal_concepts':   {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 8},
        'search_keywords':          {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 10},
        'natural_language_queries': {'type': 'array', 'items': {'type': 'string'}, 'maxItems': 5},
        'paragraph_role': {
            'type': 'string',
            'enum': ['holding', 'reasoning', 'background', 'cost',
                     'procedural', 'disposition', 'standard_of_review', 'obiter'],
        },
        'outcome_signal': {
            'type': 'string',
            'enum': ['granted', 'dismissed', 'inadmissible', 'remitted', 'partial', 'none'],
        },
    },
    'required': [
        'english_summary', 'legal_topic',
        'english_legal_concepts', 'search_keywords',
        'natural_language_queries', 'paragraph_role', 'outcome_signal',
    ],
    'additionalProperties': False,
}

SYSTEM_PROMPT = (
    'You are a Swiss legal analyst. The user gives you a paragraph from a Swiss '
    'Federal Tribunal decision in German, French, or Italian. '
    'Translate every concept into precise English legal terminology and emit '
    'structured JSON to power English-language semantic-search RAG. '
    'Be concrete: prefer \'extension of pretrial detention based on flight risk\' '
    'over \'detention\'. Output ONLY the JSON object, no preamble.'
)

print('Schema fields:', list(RAG_SCHEMA['properties'].keys()))


## 4 · Helper functions

In [6]:
import json
from typing import Iterator


def build_user_message(card: dict) -> str:
    text       = card.get('text_excerpt_original', '')[:2500]
    citation   = card.get('citation', '')
    legal_area = card.get('legal_area', '')
    existing   = card.get('issue_labels_en') or []
    parts = [
        f'Citation: {citation}',
        f'Legal area (deterministic): {legal_area}',
    ]
    if existing:
        labels = ', '.join(existing[:8])
        parts.append(f'Existing labels: {labels}')
    parts += ['', 'Paragraph (original language):', text, '', 'Produce the JSON now.']
    return '\n'.join(parts)


def _stub(summary, topic, concepts, keywords, role, method):
    return {
        'english_summary':          summary,
        'legal_topic':              topic,
        'legal_question':           '',
        'legal_rule':               '',
        'court_holding':            '',
        'factual_context':          '',
        'english_legal_concepts':   concepts,
        'search_keywords':          keywords,
        'natural_language_queries': [],
        'paragraph_role':           role,
        'outcome_signal':           'none',
        'method':                   method,
    }


def auto_classify(card: dict) -> dict | None:
    if card.get('is_notification_paragraph'):
        return _stub(
            'Procedural notification of the judgment to the parties.',
            'judgment notification',
            ['service of judgment'],
            ['notification', 'service', 'judgment communication'],
            role='procedural', method='auto_notification',
        )
    text = card.get('text_excerpt_original', '') or ''
    if len(text) < 50:
        return _stub(
            'Short procedural fragment (cross-reference or one-line ruling).',
            'procedural fragment',
            [], [], role='procedural', method='auto_short',
        )
    if COST_PROC_RE.search(text[:400]):
        return _stub(
            'Court-cost or procedural-fee allocation paragraph.',
            'court costs and procedural fees',
            ['court costs', 'procedural fees', 'legal aid'],
            ['costs', 'court fees', 'frais judiciaires', 'Gerichtskosten'],
            role='cost', method='auto_cost',
        )
    return None


def stream_input(path: Path, start_offset: int) -> Iterator[tuple[int, dict]]:
    with path.open(encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i < start_offset or not line.strip():
                continue
            try:
                yield i, json.loads(line)
            except json.JSONDecodeError:
                continue


def count_lines(path: Path) -> int:
    n = 0
    with path.open('rb') as f:
        for _ in f:
            n += 1
    return n


print('Helpers loaded.')

Helpers loaded.


## 5 · Load model with vLLM

The notebook follows the vLLM Qwen3.5 recipe:

- Uses vLLM offline inference (`LLM` + `llm.generate`) instead of Transformers `model.generate`.
- Uses Qwen's `reasoning_parser='qwen3'` when the installed vLLM build accepts that engine argument.
- Uses latency-focused settings from the recipe: MTP-1 speculative decoding and disabled prefix caching.
- Uses `StructuredOutputsParams(json=RAG_SCHEMA)` on current vLLM builds, with a fallback to `GuidedDecodingParams` on older builds.
- Disables thinking at prompt-rendering time with `enable_thinking=False`.


In [ ]:
import vllm
print('vLLM version:', getattr(vllm, '__version__', 'unknown'))


In [ ]:
# Optional runtime sanity check.
if IN_COLAB:
    !nvidia-smi


In [ ]:
# No Transformers model/tokenizer install is needed for inference.
# vLLM owns model loading, scheduling, KV cache, batching, and generation.


In [ ]:
import gc
import inspect
import os

from vllm import LLM, SamplingParams

# Force a clean multiprocessing start to reduce Jupyter/Colab worker crashes.
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

# Current vLLM structured-output API. Older builds may only expose GuidedDecodingParams.
try:
    from vllm.sampling_params import StructuredOutputsParams
except Exception:
    StructuredOutputsParams = None

try:
    from vllm.sampling_params import GuidedDecodingParams
except Exception:
    GuidedDecodingParams = None


def _call_accepts(callable_obj, name: str) -> bool:
    """Return True if callable accepts a parameter or arbitrary **kwargs."""
    try:
        sig = inspect.signature(callable_obj)
    except (TypeError, ValueError):
        return True
    params = sig.parameters
    return (
        name in params
        or any(p.kind == inspect.Parameter.VAR_KEYWORD for p in params.values())
    )


def make_sampling_params(schema: dict) -> SamplingParams:
    """Create SamplingParams using the newest vLLM structured-output API available."""
    kwargs = {
        "temperature": TEMPERATURE,
        "max_tokens": MAX_TOKENS,
    }

    if _call_accepts(SamplingParams, "structured_outputs") and StructuredOutputsParams is not None:
        kwargs["structured_outputs"] = StructuredOutputsParams(json=schema)
        print("Structured output mode: StructuredOutputsParams(json=RAG_SCHEMA)")
    elif _call_accepts(SamplingParams, "guided_decoding") and GuidedDecodingParams is not None:
        kwargs["guided_decoding"] = GuidedDecodingParams(json=schema)
        print("Structured output mode: GuidedDecodingParams(json=RAG_SCHEMA) [legacy fallback]")
    else:
        print("WARNING: installed vLLM exposes no structured-output parameter; JSON will be prompt-only.")

    return SamplingParams(**kwargs)


def build_llm_kwargs() -> dict:
    """Build LLM kwargs and only include optional recipe args supported by this vLLM build."""
    kwargs = {
        "model": MODEL_ID,
        "quantization": QUANTIZATION,
        "dtype": "bfloat16",
        "gpu_memory_utilization": GPU_MEMORY_UTIL,
        "max_model_len": MAX_MODEL_LEN,
        "trust_remote_code": False,
        "tensor_parallel_size": TENSOR_PARALLEL,
        "enforce_eager": True,
    }

    optional = {
        # vLLM Qwen3.5 recipe: use qwen3 reasoning parser.
        "reasoning_parser": "qwen3",

        # vLLM recipe: latency-focused serving uses MTP-1 and disabled prefix caching.
        "enable_prefix_caching": ENABLE_PREFIX_CACHING,
        "speculative_config": {"method": "mtp", "num_speculative_tokens": MTP_TOKENS}
            if USE_MTP else None,

        # Text-only workload: Python equivalent of --language-model-only when available.
        "language_model_only": LANGUAGE_MODEL_ONLY,

        # Fallback limiter if language_model_only is not available.
        "limit_mm_per_prompt": {"image": 0, "video": 0},
    }

    for key, value in optional.items():
        if value is None:
            continue
        if _call_accepts(LLM, key):
            kwargs[key] = value
        else:
            print(f"Skipping unsupported LLM arg: {key}")

    return kwargs


gc.collect()

print("Loading Qwen3.5 with vLLM offline inference...")
print(f"MODEL_ID={MODEL_ID}")
print(f"max_model_len={MAX_MODEL_LEN}, gpu_memory_utilization={GPU_MEMORY_UTIL}, batch_size={BATCH_SIZE}")

llm = LLM(**build_llm_kwargs())
tokenizer = llm.get_tokenizer()
sampling_params = make_sampling_params(RAG_SCHEMA)

NO_THINK = {"enable_thinking": False}

print("Model loaded successfully.")


## 6 · Run enrichment

In [ ]:
from tqdm.auto import tqdm
import json

if not INPUT_FILE.exists():
    raise FileNotFoundError(f'Input not found: {INPUT_FILE}')

start = 0
if CHECKPOINT_FILE.exists():
    try:
        start = int(CHECKPOINT_FILE.read_text().strip() or '0')
    except ValueError:
        start = 0
print(f'Resuming at line {start:,}')

total = count_lines(INPUT_FILE)
target_total = min(total, start + LIMIT) if LIMIT else total
print(f'Total={total:,}  To process={target_total - start:,}')


def render_prompt(card: dict) -> str:
    """Render Qwen chat prompt. Disable thinking where the tokenizer template supports it."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': build_user_message(card)},
    ]

    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
            enable_thinking=False,
        )
    except TypeError:
        # Older tokenizers may not expose enable_thinking. Keep a prompt-level guard.
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        return prompt + "\nDo not output chain-of-thought. Output only the JSON object.\n"


def clean_model_json(raw: str) -> str:
    raw = (raw or '').strip()

    # Defensive cleanup for non-structured-output fallback or older models.
    if raw.startswith("```json"):
        raw = raw[7:]
    if raw.startswith("```"):
        raw = raw[3:]
    if raw.endswith("```"):
        raw = raw[:-3]

    raw = raw.strip()

    # If reasoning leaked despite enable_thinking=False, drop it before parsing.
    if "</think>" in raw:
        raw = raw.split("</think>", 1)[1].strip()

    return raw


def normalize_enrichment(enriched: dict) -> dict:
    """Fill optional fields expected by downstream code and tag the method."""
    defaults = _stub('', '', [], [], role='reasoning', method='qwen35_35b_a3b_vllm')
    for key, value in defaults.items():
        enriched.setdefault(key, value)
    enriched['method'] = 'qwen35_35b_a3b_vllm'
    return enriched


out_f = OUTPUT_FILE.open('a', encoding='utf-8')
pbar = tqdm(total=target_total, initial=start, desc='enrich', unit='card', smoothing=0.05)
pending = []
json_errors = 0

def flush_batch():
    global pending, json_errors
    if not pending:
        return

    prompts = [render_prompt(card) for _, card in pending]

    # vLLM offline batched inference.
    outputs = llm.generate(
        prompts,
        sampling_params=sampling_params,
        use_tqdm=False,
    )

    for (line_idx, card), output in zip(pending, outputs):
        raw = clean_model_json(output.outputs[0].text)

        try:
            enriched = json.loads(raw)
            if not isinstance(enriched, dict):
                raise ValueError(f"Model output a {type(enriched).__name__} instead of a dict.")
            enriched = normalize_enrichment(enriched)

        except (json.JSONDecodeError, ValueError) as e:
            json_errors += 1
            enriched = _stub('', '', [], [], role='reasoning', method='json_parse_failed')
            enriched['raw_output'] = raw[:400]
            enriched['parse_error'] = str(e)[:200]

        card['rag_enrichment'] = enriched
        out_f.write(json.dumps(card, ensure_ascii=False) + '\n')

    out_f.flush()
    CHECKPOINT_FILE.write_text(str(pending[-1][0] + 1))
    pbar.update(len(pending))
    pending.clear()


# --- Main Loop ---
processed = 0
try:
    for line_idx, card in stream_input(INPUT_FILE, start):
        if LIMIT and processed >= LIMIT:
            break

        auto = auto_classify(card)
        if auto is not None:
            card['rag_enrichment'] = auto
            out_f.write(json.dumps(card, ensure_ascii=False) + '\n')
            CHECKPOINT_FILE.write_text(str(line_idx + 1))
            pbar.update(1)
            processed += 1
            continue

        pending.append((line_idx, card))
        if len(pending) >= BATCH_SIZE:
            flush_batch()
        processed += 1

    flush_batch()

finally:
    out_f.close()
    pbar.close()

print(f'Done. JSON parse errors: {json_errors}')
print(f'Output → {OUTPUT_FILE}')


## 7 · Verify output

In [ ]:
from collections import Counter

roles    = Counter()
outcomes = Counter()
methods  = Counter()
total_out = 0
missing_required = 0

REQUIRED = RAG_SCHEMA['required']

with OUTPUT_FILE.open(encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        card = json.loads(line)
        e    = card.get('rag_enrichment', {})
        total_out += 1
        roles[e.get('paragraph_role', 'MISSING')]    += 1
        outcomes[e.get('outcome_signal', 'MISSING')] += 1
        methods[e.get('method', 'MISSING')]          += 1
        if any(k not in e for k in REQUIRED):
            missing_required += 1

print(f'Total output cards : {total_out:,}')
print(f'Missing required   : {missing_required}')
print()
print('paragraph_role distribution:')
for k, v in roles.most_common():
    print(f'  {k:<25} {v:>6}  ({v/total_out*100:.1f}%)')
print()
print('outcome_signal distribution:')
for k, v in outcomes.most_common():
    print(f'  {k:<25} {v:>6}  ({v/total_out*100:.1f}%)')
print()
print('method distribution:')
for k, v in methods.most_common():
    print(f'  {k:<25} {v:>6}  ({v/total_out*100:.1f}%)')

In [ ]:
# Show 3 random LLM-enriched cards for a quick quality check
import random

llm_cards = []
with OUTPUT_FILE.open(encoding='utf-8') as f:
    for line in f:
        if not line.strip():
            continue
        card = json.loads(line)
        if card.get('rag_enrichment', {}).get('method', '').startswith('qwen35'):
            llm_cards.append(card)

for card in random.sample(llm_cards, min(3, len(llm_cards))):
    e = card['rag_enrichment']
    print('─' * 70)
    print('Citation     :', card.get('citation', ''))
    print('Role         :', e.get('paragraph_role'))
    print('Outcome      :', e.get('outcome_signal'))
    print('Topic        :', e.get('legal_topic'))
    print('Summary      :', e.get('english_summary', '')[:200])
    print('Keywords     :', e.get('search_keywords'))
    print('NL queries   :', e.get('natural_language_queries'))
    print()